# Experiment 08: DSPy Routing & Prompt Optimization Lab

**Offline lab, not a runtime dependency.** This notebook never edits `src/`. It reads a
snapshot of the model-facing tool catalog, compiles a next-action router against the routing
benchmark, and exports candidate text for human review.

The catalog snapshot is generated outside Jupyter by
`experiments/scripts/refresh_catalog_snapshot.py`. Regenerate it whenever the model-facing
tool list changes; the notebook asserts that the snapshot matches the agent's allowlist.

## What this lab optimizes

| Artifact | Optimized here? | How |
|---|---|---|
| Next-action routing policy | ✅ | `dspy.GEPA` over a `ChainOfThought` router |
| Tool descriptions | ✅ (tournament) | Contract-first / contrastive variants scored on the benchmark |
| Budget ledger, quarantine, verification, read sandbox | ❌ | These are code constraints; never let an optimizer rewrite them |

## Setup

Use a **dedicated kernel** so the recipe's pinned `openai==1.65.1` is not upgraded:

```bash
python -m venv .venv-optim
.venv-optim\Scripts\python -m pip install -r experiments/requirements-optim.txt
.venv-optim\Scripts\python -m ipykernel install --user --name bodhan-dspy --display-name "Python 3 (bodhan-dspy)"
```

Then select **Python 3 (bodhan-dspy)** in Jupyter. If `dspy` is missing, Sections 0–3 still
run and validate the dataset; Sections 4+ are skipped with a clear message.

## Cost controls

- `RUN_LM_PROBE` — one call to verify the endpoint.
- `RUN_BASELINES` — production, zero-shot, and CoT on a 12-case stratified pilot of the held-out test split (PILOT_N=12).
- `RUN_OPTIMIZATION` — `GEPA(auto="light")`, capped by `MAX_METRIC_CALLS`.
- `RUN_TOOL_DESC_TOURNAMENT` — global variant comparison; per-tool greedy is a second flag.

Every paid cell is guarded. Read the toggle cell before running all.

## Benchmark size

87 cases: 37 train / 21 dev / 29 test. Every model-facing tool and every category appears in
the test split, including cross-lingual multi-modal pipelines (speech -> English intermediate -> translation -> audio deliverable) and honest failure refusals.

> Note (2026-11-19): this notebook's embedded cell outputs were executed against the pre-v3 benchmark (the embedded `Cases: 70 | train/dev/test: 31 15 24` output and `dd3ecb58...` hash predate even the v2 rebuild) and were not re-executed. The dataset is now **v3**: 87 cases, 37/21/29 train/dev/test splits, every model-facing tool and every category in the test split, `dataset_sha256 = 2c3abeefd189128415a7efd7ff67826d8099e2c98630c39edd83b67ae92a8eb4` (recorded in `experiments/data/README.md`; `validation_report.json` and the split files are derived and regenerate byte-identically via `refresh_catalog_snapshot.py` -> `build_routing_dataset.py`).
>
> **Revision note (2026-11-19, combined pool).** Section 3b below builds a **combined decision-point pool**: v3 routing (87) + recovery v1 (15) + trajectory-derived decision points from the 25 task specs (ambiguous milestone hints included as multi-tool acceptable sets). Recovery rows are single next-tool decisions at a failure moment; the unified metric hard-fails forbidden tools and emits doctrine-aware feedback (adapt-don't-stop, checkpointing, reuse). v1 splits are assigned deterministically in-notebook (seed 20261119) - derived, like the v3 splits. Baselines (B0), the GEPA loop, and the held-out evaluation all run over the pooled TRAIN/DEV/TEST.


## 0. Before you run: regenerate derived datasets (byte-identical, one command each)

`experiments/data` holds no hand-maintained derived state: **every dataset file is fully
recreatable from a script**. The catalog cell below loads `tool_catalog_snapshot.json`,
which is derived — so regenerate before Run All (from the recipe root, recipe venv):

```bash
python experiments/scripts/refresh_catalog_snapshot.py     # tool_catalog_snapshot.json (must run outside Jupyter)
python experiments/scripts/build_routing_dataset.py        # v3 splits + validation_report.json
python experiments/scripts/build_trajectories_dataset.py   # trajectories_v1.jsonl (25 task specs)
python experiments/scripts/build_recovery_dataset.py       # recovery_cases_v1.jsonl (15 recovery decisions)
```

Each builder runs its own gates and refuses to write on any failure; regeneration is
byte-identical, and the frozen-hash assert in the dataset cell verifies `routing_cases.jsonl`
against the hash recorded in `experiments/data/README.md`. If a builder refuses to write,
fix the gate failure — never hand-edit the JSONL.

## 0. Paths, configuration, and spend toggles

The path bootstrap matches `03_mcp_server_and_tools.ipynb` so the notebook works from the
recipe root or from the repository root.

In [ ]:
from __future__ import annotations

import asyncio
import collections
import difflib
import hashlib
import json
import os
import random
import re
import statistics
import sys
import textwrap
from pathlib import Path

# Locate the recipe root by searching for the authentic sample_data anchor.
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (base / "sample_data" / "demo1_indic_nlp_paper.pdf").is_file():
        recipe_root = base
        break
    cand = base / "ref" / "Bodhan-AI-Cookbook" / "examples" / "bodhan_tool_agent"
    if (cand / "sample_data" / "demo1_indic_nlp_paper.pdf").is_file():
        recipe_root = cand
        break
    cand2 = base / "examples" / "bodhan_tool_agent"
    if (cand2 / "sample_data" / "demo1_indic_nlp_paper.pdf").is_file():
        recipe_root = cand2
        break
else:
    recipe_root = Path.cwd().resolve()

if str(recipe_root) not in sys.path:
    sys.path.insert(0, str(recipe_root))

from types import SimpleNamespace


def _load_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if path.is_file():
        for raw in path.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            value = value.split("#")[0].strip()
            values[key.strip()] = value.strip('"').strip("'")
    return values


_env = _load_env_file(recipe_root / ".env")
config = SimpleNamespace(
    llm_model=_env.get("LLM_MODEL", "gpt-4o"),
    llm_base_url=_env.get("LLM_BASE_URL", "").rstrip("/"),
    llm_api_key=_env.get("LLM_API_KEY", ""),
    llm_reasoning_effort=_env.get("LLM_REASONING_EFFORT", ""),
    base_url=_env.get("BODHAN_BASE_URL", "https://api.bodhan.ai/v1").rstrip("/"),
    translate_key=_env.get("BODHAN_TRANSLATE_KEY") or _env.get("BODHAN_API_KEY", ""),
)

# Read .env directly and avoid importing src.* in the kernel: FastMCP's
# Tool.parameters access deadlocks some Jupyter kernels on Windows.
DATA_DIR = recipe_root / "experiments" / "data"
OUT_DIR = recipe_root / "experiments" / "outputs" / "exp08_dspy_optimization"
EXPORT_DIR = OUT_DIR / "export"
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ---- spend toggles ---------------------------------------------------------
RUN_LM_PROBE = True                # one OpenAI-compatible call; 20s hard timeout; enable to verify the endpoint
RUN_BASELINES = True               # pilot: production + zero-shot + CoT
RUN_OPTIMIZATION = False           # False loads cached compile in 0.1s; True re-runs 440 GEPA rollouts
RUN_TOOL_DESC_TOURNAMENT = True    # global current/contract-first/contrastive (cheap 3-variant test)
RUN_FULL_TOOL_GREEDY = False        # per-tool coordinate descent (more calls)

PILOT_N = 12
MAX_METRIC_CALLS = 120
SEED = 7
random.seed(SEED)

print("Recipe root :", recipe_root)
print("Dataset dir :", DATA_DIR)
print("Output dir  :", OUT_DIR)
print("LLM model   :", config.llm_model)
print("LLM base URL:", config.llm_base_url or "(provider default)")
print("Bodhan base :", config.base_url)
print("\nSpend toggles:")
for name in (
    "RUN_LM_PROBE", "RUN_BASELINES", "RUN_OPTIMIZATION",
    "RUN_TOOL_DESC_TOURNAMENT", "RUN_FULL_TOOL_GREEDY",
):
    print(f"  {name:30s} = {globals()[name]}")


## 1. DSPy and the OpenAI-compatible endpoint

`dspy.LM("openai/<model>", api_base=..., api_key=...)` routes through the OpenAI-compatible
adapter. The `openai/` prefix selects the protocol, not the vendor. `api_base` must include
`/v1`; DSPy does not add it for you.

If the probe fails, check, in order: (1) the `/v1` suffix, (2) the `openai/` prefix,
(3) `model_type="chat"`, (4) whether the gateway requires a non-empty placeholder key.

In [ ]:
DSPY_AVAILABLE = False
DSPY_IMPORT_ERROR = None

try:
    import dspy

    DSPY_AVAILABLE = True
except Exception as exc:  # pragma: no cover - environment dependent
    DSPY_IMPORT_ERROR = str(exc)

lm = None
lm_hot = None

if not DSPY_AVAILABLE:
    print("DSPy is not installed in this kernel.")
    print("Install it in a dedicated venv:  pip install \"dspy[litellm]\" ipykernel")
    print("Import error:", DSPY_IMPORT_ERROR)
else:
    print("dspy", getattr(dspy, "__version__", "unknown"))
    base_kwargs = dict(
        api_base=config.llm_base_url or None,
        api_key=config.llm_api_key or None,
        cache=True,
        reasoning_effort=getattr(config, "llm_reasoning_effort", "minimal") or "minimal"
    )
    try:
        lm = dspy.LM(f"openai/{config.llm_model}", model_type="chat", timeout=60, **base_kwargs)
    except TypeError:
        try:
            # Older DSPy builds may not accept model_type or timeout.
            lm = dspy.LM(f"openai/{config.llm_model}", timeout=60, **base_kwargs)
        except TypeError:
            lm = dspy.LM(f"openai/{config.llm_model}", **base_kwargs)
    dspy.configure(lm=lm)
    print("Configured LM:", lm)

    if RUN_LM_PROBE:
        try:
            from openai import OpenAI

            probe_client = OpenAI(
                base_url=config.llm_base_url or None,
                api_key=config.llm_api_key or "probe-key",
                timeout=20.0,
            )
            probe = probe_client.chat.completions.create(
                model=config.llm_model,
                messages=[{"role": "user", "content": "Reply with exactly one word: OK"}],
            )
            print("Probe response:", probe.choices[0].message.content)
        except Exception as exc:
            print("Probe failed:", type(exc).__name__, exc)
            print("Fix the endpoint before enabling RUN_BASELINES / RUN_OPTIMIZATION.")
    else:
        print("LM probe disabled (RUN_LM_PROBE=False); no network call was made.")

## 2. The live tool catalog

The catalog is loaded from `experiments/data/tool_catalog_snapshot.json`, generated from
`agent.py::_get_openai_tool_definitions()` — the same `mcp._tool_manager.list_tools()` source
that production uses. FastMCP's `Tool.parameters` property deadlocks inside a Jupyter kernel,
so the snapshot is generated **outside** Jupyter and reloaded here. Regenerate it whenever
`server.py` changes; the JSON is the audit trail.

`submit_plan` and `finish` are agent-level control tools, so they are added from `agent.py`
and scoped by phase:

- `phase="plan"` → `submit_plan` only.
- `phase="act"` → the 8 model-facing MCP tools + `finish`.

The snapshot is generated from the agent's `MODEL_FACING_TOOLS` allowlist, so it is exactly
what the model sees. `get_cache_stats`, `clear_cache` and the deleted `chunk_audio` are
absent by construction; Section 2 asserts that here.

In [ ]:
import ast

# Extract SYSTEM_PROMPT from agent.py without importing it. Importing the
# FastMCP registry in a Jupyter kernel can deadlock on Tool.parameters.
_agent_source = (recipe_root / "src" / "bodhan_agent" / "agent.py").read_text(encoding="utf-8")
_agent_tree = ast.parse(_agent_source)
SYSTEM_PROMPT = None
for _node in _agent_tree.body:
    if isinstance(_node, ast.Assign) and any(
        isinstance(t, ast.Name) and t.id == "SYSTEM_PROMPT" for t in _node.targets
    ):
        SYSTEM_PROMPT = ast.literal_eval(_node.value)
        break
assert SYSTEM_PROMPT, "Could not extract SYSTEM_PROMPT from agent.py"

# Live catalog snapshot, generated outside Jupyter (FastMCP's Tool.parameters
# property can deadlock the kernel). Same shape as
# agent.py::_get_openai_tool_definitions(). Regenerate when server.py changes.
CATALOG_SNAPSHOT = DATA_DIR / "tool_catalog_snapshot.json"
assert CATALOG_SNAPSHOT.is_file(), f"Missing catalog snapshot: {CATALOG_SNAPSHOT}"
TOOL_SNAPSHOT = json.loads(CATALOG_SNAPSHOT.read_text(encoding="utf-8"))
MCP_TOOLS = TOOL_SNAPSHOT["mcp_tools"]
CONTROL_TOOLS = TOOL_SNAPSHOT["control_tools"]
ALL_TOOLS = {t["name"]: t for t in MCP_TOOLS + CONTROL_TOOLS}

# --- catalog contract -------------------------------------------------------
# The snapshot must equal the agent's allowlist. Parsing agent.py with ast keeps
# the kernel free of src imports: FastMCP's Tool.parameters access deadlocks
# some Jupyter kernels on Windows.
_ALLOW = None
for _node in _agent_tree.body:
    _targets = getattr(_node, "targets", None)
    if _targets is None and isinstance(_node, ast.AnnAssign):
        _targets = [_node.target]
    if not _targets:
        continue
    if not any(isinstance(t, ast.Name) and t.id == "MODEL_FACING_TOOLS" for t in _targets):
        continue
    _value = _node.value
    _ALLOW = set(ast.literal_eval(_value.args[0] if isinstance(_value, ast.Call) else _value))
    break
assert _ALLOW, "Could not extract MODEL_FACING_TOOLS from agent.py"
_ADVERTISED = {t["name"] for t in MCP_TOOLS}
assert _ADVERTISED == _ALLOW, (
    f"snapshot/allowlist mismatch: {sorted(_ADVERTISED ^ _ALLOW)}"
)
for _operator_only in ("get_cache_stats", "clear_cache", "chunk_audio"):
    assert _operator_only not in _ADVERTISED, (
        f"{_operator_only} must not be model-facing"
    )
print("Catalog contract: OK --", len(_ALLOW), "model-facing tools")


def catalog_for_phase(phase: str) -> list[dict]:
    if phase == "plan":
        return [ALL_TOOLS["submit_plan"]]
    return [t for t in MCP_TOOLS] + [ALL_TOOLS["finish"]]


def openai_tool_spec(tool: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool["description"],
            "parameters": tool["parameters"],
        },
    }


def render_catalog(phase: str, overrides: dict[str, str] | None = None) -> str:
    overrides = overrides or {}
    rendered = []
    for tool in catalog_for_phase(phase):
        item = dict(tool)
        item["description"] = overrides.get(tool["name"], tool["description"])
        rendered.append(item)
    return json.dumps(rendered, ensure_ascii=False, indent=2)


print(f"MCP tools: {len(MCP_TOOLS)} | control tools: {len(CONTROL_TOOLS)}")
for tool in MCP_TOOLS:
    first = tool["description"].splitlines()[0] if tool["description"] else "(no description)"
    print(f"  - {tool['name']:24s} {first[:90]}")
print("\nplan catalog:", [t["name"] for t in catalog_for_phase("plan")])
print("act catalog :", [t["name"] for t in catalog_for_phase("act")])
print("snapshot   :", CATALOG_SNAPSHOT)

## 3. Load and validate the routing benchmark

`experiments/data/routing_cases.jsonl` contains 87 hand-authored decision points, built and
checked by `experiments/scripts/build_routing_dataset.py`. Each case is **one next-action
decision**, not a whole trajectory. `history_summary` supplies prior observations;
`acceptable_next_tools` is always exactly one tool, because hedging hides routing errors;
`arg_constraints` records only arguments the task makes unambiguous.

The test split hash is frozen in `validation_report.json`. Do not tune against it.

In [ ]:
DATASET = DATA_DIR / "routing_cases.jsonl"
assert DATASET.is_file(), f"Missing dataset: {DATASET}"

rows = [json.loads(line) for line in DATASET.read_text(encoding="utf-8").splitlines() if line.strip()]
by_id = {r["id"]: r for r in rows}

REQUIRED = {
    "id", "phase", "task", "context_files", "history_summary", "expected_next_tool",
    "acceptable_next_tools", "arg_constraints", "category", "difficulty", "rationale",
    "split", "source_slice", "provenance", "schema_version",
}
issues = []
for r in rows:
    missing = REQUIRED - set(r)
    if missing:
        issues.append(f"{r.get('id')}: missing {sorted(missing)}")
    if r["phase"] not in {"plan", "act"}:
        issues.append(f"{r['id']}: bad phase {r['phase']}")
    if r["expected_next_tool"] not in ALL_TOOLS:
        issues.append(f"{r['id']}: unknown tool {r['expected_next_tool']}")
    if r["expected_next_tool"] not in r["acceptable_next_tools"]:
        issues.append(f"{r['id']}: expected not in acceptable set")
    if r["phase"] == "plan" and r["expected_next_tool"] != "submit_plan":
        issues.append(f"{r['id']}: plan phase must use submit_plan")
    if r["phase"] == "act" and r["expected_next_tool"] == "submit_plan":
        issues.append(f"{r['id']}: act phase must not use submit_plan")
    for path in r.get("context_files") or []:
        if not (recipe_root / path).exists():
            issues.append(f"{r['id']}: missing context file {path}")

    # Tool-name leakage: no advertised tool name may appear inside the task text.
    task_lc = str(r.get("task") or "").lower()
    for tool_name in _ADVERTISED:
        if tool_name and tool_name in task_lc:
            issues.append(f"{r['id']}: task leaks tool name {tool_name}")

    # With prior work in history, the text to transform must be unambiguous.
    if (r.get("history_summary") and
            r["expected_next_tool"] in {"translate_text", "synthesize_speech", "write_text_file"}):
        if not ({"text", "content"} & set(r.get("arg_constraints") or {})):
            issues.append(f"{r['id']}: missing text/content arg constraint for {r['expected_next_tool']}")

    # Bounded .txt reads must start inside the file; skip .docx/binary files.
    if r["expected_next_tool"] == "read_text_file":
        ac = r.get("arg_constraints") or {}
        rel = ac.get("file_path") or ac.get("path") or ac.get("file")
        if isinstance(rel, str) and rel.lower().endswith(".txt"):
            src = recipe_root / rel
            if src.is_file():
                try:
                    file_text = src.read_text(encoding="utf-8", errors="replace")
                    offset = ac.get("offset", 0)
                    if isinstance(offset, int) and offset >= len(file_text):
                        issues.append(f"{r['id']}: offset {offset} >= len(file_text) {len(file_text)}")
                except (OSError, ValueError):
                    pass
if issues:
    raise AssertionError("Dataset validation failed:\n" + "\n".join(issues))

TRAIN = [r for r in rows if r["split"] == "train"]
DEV = [r for r in rows if r["split"] == "dev"]
TEST = [r for r in rows if r["split"] == "test"]

assert {r["expected_next_tool"] for r in rows} <= set(ALL_TOOLS), (
    "dataset routes to a tool that is not model-facing: "
    f"{sorted({r['expected_next_tool'] for r in rows} - set(_ALLOW))}"
)
assert {r["expected_next_tool"] for r in TEST} == set(ALL_TOOLS), (
    "every model-facing tool must appear in the test split"
)
assert {r["category"] for r in TEST} == {r["category"] for r in rows}, (
    "every category must appear in the test split"
)

print("Cases:", len(rows), "| train/dev/test:", len(TRAIN), len(DEV), len(TEST))
print("Phase:", dict(collections.Counter(r["phase"] for r in rows)))
print("Tools:", dict(collections.Counter(r["expected_next_tool"] for r in rows)))
print("Categories:", dict(collections.Counter(r["category"] for r in rows)))

# Coverage matrix: category x tool (non-zero cells only).
cats = sorted({r["category"] for r in rows})
tools = sorted({r["expected_next_tool"] for r in rows})
print("\nCoverage matrix (category x tool):")
print("  " + " ".join(f"{t[:11]:>11s}" for t in tools))
for cat in cats:
    counts = collections.Counter(r["expected_next_tool"] for r in rows if r["category"] == cat)
    print(f"{cat[:18]:18s} " + " ".join(f"{counts.get(t, 0):11d}" for t in tools))

# validation_report.json is derived (deleted 2026-11-19); the frozen hash lives in README.md.
frozen_hash = None
for _line in (DATA_DIR / "README.md").read_text(encoding="utf-8").splitlines():
    if "dataset_sha256" in _line and "=" in _line:
        _candidate = _line.split("=", 1)[1].strip().split("`")[0].strip()
        if len(_candidate) == 64:
            frozen_hash = _candidate
            break
assert frozen_hash, "Could not parse dataset_sha256 from README.md (Directory map section)"
live_hash = hashlib.sha256(DATASET.read_bytes()).hexdigest()
print("\nFrozen hash (README):", frozen_hash)
print("Live hash   :", live_hash)
print("Hash match  :", live_hash == frozen_hash)
assert live_hash == frozen_hash, "routing_cases.jsonl does not match the frozen hash recorded in README.md"

print("\nSample cases:")
for r in (rows[0], next(x for x in rows if x["phase"] == "plan"), rows[-1]):
    print(json.dumps({k: r[k] for k in ("id", "phase", "task", "expected_next_tool", "acceptable_next_tools", "arg_constraints")}, ensure_ascii=False, indent=2))

## 3b. Combined decision-point pool (v3 + recovery + trajectories)

All three datasets are **routing problems at the decision level**, so they are optimized together:

- **v3 routing (87)** — unchanged, frozen splits.
- **Recovery v1 (15)** — mechanically mapped: `preferred_next_tool` becomes `expected_next_tool`;
  the failure observation already lives in `history_summary`; `forbidden_next_tools` hard-fail with
  doctrine-aware feedback (adapt-don't-stop).
- **Trajectories v1 (25 specs)** — decomposed into per-milestone decision points. Gold comes from
  the milestone `tool_hints` (advisory): ambiguous hints become a **multi-tool acceptable set**
  (include-ambiguous decision, 2026-11-19); hint-less milestones are skipped and counted.

v1 splits are assigned **deterministically in this notebook** (seed 20261119, stratified per
category; categories with n>=3 keep at least one held-out case) — derived, like the v3 splits.
The pool cell rebinds `rows` / `by_id` / `TRAIN` / `DEV` / `TEST`, so every later cell
(B0 baseline, GEPA compile, held-out evaluation) runs over the combined pool. Per-suite
breakdowns are printed so routing, recovery, and trajectory behavior are always visible separately.

In [ ]:
# --- 3b. Combined decision-point pool: v3 routing + recovery v1 + trajectory-derived points ---
import random as _random

V1_SEED = 20261119
recovery_path = DATA_DIR / "recovery_cases_v1.jsonl"
traj_path = DATA_DIR / "trajectories_v1.jsonl"
assert recovery_path.is_file(), f"Missing {recovery_path}"
assert traj_path.is_file(), f"Missing {traj_path}"

recovery_rows = [json.loads(l) for l in recovery_path.read_text(encoding="utf-8").splitlines() if l.strip()]
traj_rows = [json.loads(l) for l in traj_path.read_text(encoding="utf-8").splitlines() if l.strip()]


def assign_v1_splits(cases, seed):
    # Deterministic per-category split: fixed-seed shuffle; categories with n >= 3 keep at
    # least one held-out (test) case, n >= 4 keeps a dev case; the rest go to train.
    rng = _random.Random(seed)
    by_cat = {}
    for c in cases:
        by_cat.setdefault(c["category"], []).append(c["id"])
    assignment = {}
    for cat in sorted(by_cat):
        ids = sorted(by_cat[cat])
        rng.shuffle(ids)
        n = len(ids)
        n_test = 1 if n >= 3 else 0
        n_dev = 1 if n >= 4 else 0
        for i, rid in enumerate(ids):
            assignment[rid] = "test" if i < n_test else ("dev" if i < n_test + n_dev else "train")
    return assignment


recovery_split = assign_v1_splits(recovery_rows, V1_SEED)
traj_split = assign_v1_splits(traj_rows, V1_SEED + 1)

pool = []
for r in rows:  # v3 rows, already validated above
    pool.append({**r, "suite": "routing", "forbidden_next_tools": []})

for r in recovery_rows:
    pool.append({
        "id": r["id"], "phase": r["phase"], "task": r["task"],
        "context_files": r.get("context_files") or [],
        "history_summary": r["history_summary"],
        "expected_next_tool": r["preferred_next_tool"],
        "preferred_next_tool": r["preferred_next_tool"],
        "acceptable_next_tools": list(dict.fromkeys([r["preferred_next_tool"]] + list(r["acceptable_next_tools"]))),
        "forbidden_next_tools": list(r.get("forbidden_next_tools") or []),
        "arg_constraints": r.get("arg_constraints") or {},
        "category": r["category"], "difficulty": r["difficulty"],
        "rationale": r["rationale"], "split": recovery_split[r["id"]],
        "suite": "recovery",
        "observed_failure": r.get("observed_failure"),
        "expected_recovery_policy": r.get("expected_recovery_policy"),
        "source_slice": "recovery_cases_v1", "provenance": r.get("provenance"),
        "schema_version": r.get("schema_version", "1.0"),
    })

derived, skipped_no_hints = 0, 0
for t in traj_rows:
    base_split = traj_split[t["id"]]
    completed = []
    for mi, m in enumerate(t["milestones"], start=1):
        hints = [h for h in (m.get("tool_hints") or []) if h]
        if not hints:
            skipped_no_hints += 1
            completed.append(m["name"])
            continue
        distinct = list(dict.fromkeys(hints))
        hist_lines = [f"- completed milestone: {name}" for name in completed]
        if completed:
            hist_lines.append("- intermediate results checkpointed to artifacts/")
        hist_lines.append(f"- next milestone: {m['name']} - {m['description']}")
        rubric = t.get("rubric") or {}
        if rubric.get("max_wasted_calls") is not None:
            hist_lines.append(
                f"- waste budget: at most {rubric['max_wasted_calls']} wasted calls"
                f" ({rubric.get('budget_efficiency') or 'repeating paid work already in history'})"
            )
        pool.append({
            "id": f"{t['id']}#m{mi}", "phase": "act", "task": t["task"],
            "context_files": list(t.get("input_files") or []),
            "history_summary": chr(10).join(hist_lines),
            "expected_next_tool": distinct[0],
            "acceptable_next_tools": distinct,
            "forbidden_next_tools": [],
            "arg_constraints": {},
            "category": t["category"], "difficulty": t["difficulty"],
            "rationale": (f"Derived from {t['id']} milestone {mi} ({m['name']}); tool hints are"
                          " advisory gold (trajectories README design rules)."),
            "split": base_split, "suite": "trajectory",
            "source_slice": f"{t['id']}#m{mi}", "provenance": t.get("provenance"),
            "schema_version": "1.0",
        })
        derived += 1
        completed.append(m["name"])

# --- validate the pool ---
issues = []
for r in pool:
    if r["expected_next_tool"] not in r["acceptable_next_tools"]:
        issues.append(f"{r['id']}: expected not in acceptable")
    if set(r.get("forbidden_next_tools") or []) & set(r["acceptable_next_tools"]):
        issues.append(f"{r['id']}: forbidden intersects acceptable")
    if r["expected_next_tool"] not in ALL_TOOLS:
        issues.append(f"{r['id']}: unknown tool {r['expected_next_tool']}")
    for path in r.get("context_files") or []:
        if not (recipe_root / path).exists():
            issues.append(f"{r['id']}: missing context file {path}")
    task_lc = str(r.get("task") or "").lower()
    for tool_name in _ADVERTISED:
        if tool_name and tool_name in task_lc:
            issues.append(f"{r['id']}: task leaks tool name {tool_name}")
_ids = [r["id"] for r in pool]
if len(_ids) != len(set(_ids)):
    issues.append("duplicate ids in pool")
if issues:
    raise AssertionError("Combined pool validation failed:\n" + "\n".join(issues))

# --- rebind the names every later cell consumes (baselines, GEPA, held-out eval) ---
rows = pool
by_id = {r["id"]: r for r in pool}
TRAIN = [r for r in pool if r["split"] == "train"]
DEV = [r for r in pool if r["split"] == "dev"]
TEST = [r for r in pool if r["split"] == "test"]

pool_hash = hashlib.sha256(
    ("\n".join(json.dumps(r, sort_keys=True, ensure_ascii=False) for r in pool)).encode("utf-8")
).hexdigest()

print("Combined pool:", len(pool), "decision points |", dict(collections.Counter(r["suite"] for r in pool)))
print("Splits per suite:")
for suite in ("routing", "recovery", "trajectory"):
    sub = [r for r in pool if r["suite"] == suite]
    print(f"  {suite:11s}", dict(collections.Counter(r["split"] for r in sub)), f"(n={len(sub)})")
print("Trajectory derivation: derived", derived, "decision points; skipped", skipped_no_hints, "hint-less milestones")
print("Pool composition hash (record in reports):", pool_hash)

## 4. Router signature, programs, and metric

The router is intentionally **not** a `dspy.ReAct` loop. It is a single decision step that can
be evaluated and optimized cheaply. The production loop keeps its budget circuit breaker,
untrusted-data quarantine, verification, and tracing; only the decision text is a candidate
for replacement.

The metric is shaped for GEPA: it returns `dspy.Prediction(score, feedback)`. The score is
0.7 tool choice + 0.2 argument agreement + 0.1 concise reasoning. Phase violations and invalid
`finish` statuses are hard penalties.

In [ ]:
parse_args = None
to_example = None
score_case = None
gepa_metric = None
float_metric = None
Router = None

if DSPY_AVAILABLE:
    def parse_args(raw) -> dict:
        if isinstance(raw, dict):
            return raw
        if not raw:
            return {}
        text = str(raw).strip()
        if text.startswith("```"):
            text = re.sub(r"^```[a-zA-Z]*\s*", "", text)
            text = re.sub(r"\s*```$", "", text)
        try:
            value = json.loads(text)
            return value if isinstance(value, dict) else {}
        except Exception:
            match = re.search(r"\{.*\}", text, re.S)
            if match:
                try:
                    value = json.loads(match.group(0))
                    return value if isinstance(value, dict) else {}
                except Exception:
                    return {}
        return {}

    class RouteTool(dspy.Signature):
        """You are the next-action router for an Indic media and document agent.

        Select exactly one tool from the catalog that makes direct progress on the user's task.
        Never select a tool outside the catalog. In phase 'plan', only submit_plan is available;
        in phase 'act', submit_plan is unavailable.

        DELIVERABLE & BOUNDARY CONTRACTS:
        - Bounded character ranges/slices (e.g. characters 4,000–5,999) on ANY file (including .docx): always call read_text_file with offset=start, limit=end-start+1. NEVER call extract_document_text for bounded windows.
        - Audio deliverables or voiceovers (.wav files): call synthesize_speech directly. NEVER call write_text_file for audio.
        - Audio transcription or direct English speech translation: call transcribe_audio directly; it auto-chunks audio >30s. NEVER call inspect_media first when transcription is requested.
        - Audio verification: when tasked with verifying a generated audio artifact (.wav), call transcribe_audio to verify speech fidelity.
        - Media metadata queries (duration, format, dimensions, page count): call inspect_media ONLY when metadata is explicitly requested without content processing.
        - Plain text & code files (.txt, .md, .csv, .json, .srt, .py): always call read_text_file. NEVER use extract_document_text (OCR) on raw text files.
        - Translation: translate_text supports 28 Indic languages and English. Specify exact script subtags (ks-Deva for Kashmiri, sd-Arab for Sindhi, mni-Beng for Manipuri, ta for Tamil, mr for Marathi). Omit source_language.
        - Voice cloning is infeasible: synthesize_speech does not clone voices from reference audio (.ogg, .wav); in phase 'plan', submit_plan with feasible=False.
        - Security & Refusals: Refuse .env/keys or path traversal (..) with finish(status='failed'). Refuse missing files or unsupported non-Indic languages with finish(status='impossible').
        - History continuity: When history confirms an upstream extraction or transcript already exists, proceed with the next downstream step without repeating completed work.
        Return arguments as a clean JSON object adhering strictly to the selected tool's schema."""

        task: str = dspy.InputField(desc="The user's task, verbatim.")
        phase: str = dspy.InputField(desc="'plan' or 'act'.")
        history: str = dspy.InputField(desc="Prior tool calls and observations; empty on the first act step.")
        tool_catalog: str = dspy.InputField(desc="JSON array of available tools with name, description, and parameters.")
        tool_name: str = dspy.OutputField(desc="Exact tool name from the catalog.")
        arguments_json: str = dspy.OutputField(desc="A JSON object of arguments, no markdown fences.")

    class Router(dspy.Module):
        def __init__(self, use_cot: bool = True):
            super().__init__()
            self.predict = dspy.ChainOfThought(RouteTool) if use_cot else dspy.Predict(RouteTool)

        def forward(self, task: str, phase: str, history: str, tool_catalog: str):
            pred = self.predict(task=task, phase=phase, history=history, tool_catalog=tool_catalog)
            return dspy.Prediction(
                tool_name=str(pred.tool_name).strip().strip('"').strip("'"),
                arguments_json=getattr(pred, "arguments_json", "{}"),
                reasoning=getattr(pred, "reasoning", ""),
            )

    def to_example(case: dict, overrides: dict[str, str] | None = None):
        return dspy.Example(
            task=case["task"],
            phase=case["phase"],
            history=case["history_summary"],
            tool_catalog=render_catalog(case["phase"], overrides),
            case_id=case["id"],
            expected_next_tool=case["expected_next_tool"],
            acceptable_next_tools=tuple(case["acceptable_next_tools"]),
            forbidden_next_tools=tuple(case.get("forbidden_next_tools") or ()),
            suite=case.get("suite", "routing"),
            preferred_next_tool=case.get("preferred_next_tool"),
            expected_recovery_policy=case.get("expected_recovery_policy"),
            arg_constraints=case["arg_constraints"],
            category=case["category"],
        ).with_inputs("task", "phase", "history", "tool_catalog")

    def _field(obj, name, default=None):
        if isinstance(obj, dict):
            return obj.get(name, default)
        return getattr(obj, name, default)

    PAID_TOOLS = {"transcribe_audio", "translate_text", "synthesize_speech", "extract_document_text"}

    def _policy_note(gold):
        pol = _field(gold, "expected_recovery_policy", None) or {}
        cls = pol.get("classify_as")
        if cls == "transient":
            return ("This failure is transient transport trouble - weather, not a verdict. "
                    "Adapt: retry with a different shape (e.g. smaller chunks), checkpoint "
                    "progress with write_text_file, or route around. Never finish-fail a "
                    "milestone because the network blinked.")
        if cls == "capability":
            return "Capability limit: decline honestly (finish impossible / plan infeasible) without spending."
        if cls == "permanent":
            return "Permanent failure: finish honestly with the real status and preserved work."
        return "Classify the failure before reacting; match the recovery to the classification."

    def score_case(gold, pred) -> tuple[float, str, bool]:
        gold_tool = _field(gold, "expected_next_tool", "")
        acceptable = set(_field(gold, "acceptable_next_tools", ()) or ())
        constraints = _field(gold, "arg_constraints", {}) or {}
        phase = _field(gold, "phase", "act")
        tool = str(_field(pred, "tool_name", "") or "").strip()
        args = parse_args(_field(pred, "arguments_json", "{}"))
        reasoning = str(_field(pred, "reasoning", "") or "")

        feedback = []
        unsafe = False
        score = 0.0

        if phase == "act" and tool == "submit_plan":
            return 0.0, "submit_plan is unavailable in phase=act", True
        if phase == "plan" and tool != "submit_plan":
            return 0.0, "phase=plan must use submit_plan", True

        forbidden = set(_field(gold, "forbidden_next_tools", ()) or ())
        suite = _field(gold, "suite", "routing")
        preferred = _field(gold, "preferred_next_tool", None)
        if tool in forbidden:
            return 0.0, f"FORBIDDEN after this observation: {tool}. {_policy_note(gold)}", True
        if preferred and tool == preferred:
            score += 0.7
        elif tool in acceptable:
            score += 0.56
            feedback.append("defensible alternative, but not the preferred repair")
        else:
            feedback.append(f"expected one of {sorted(acceptable)}, got {tool or 'none'}")
            if suite == "recovery":
                feedback.append(_policy_note(gold))
            return 0.0, "; ".join(feedback), unsafe

        if constraints:
            matched = 0
            for key, expected in constraints.items():
                if key not in args:
                    feedback.append(f"missing arg {key}")
                    continue
                got = args[key]
                if isinstance(got, str) and isinstance(expected, str):
                    same = got.strip() == expected.strip()
                else:
                    same = got == expected
                if same:
                    matched += 1
                else:
                    feedback.append(f"arg {key}: expected {expected!r}, got {got!r}")
            score += 0.2 * (matched / len(constraints))
        else:
            score += 0.2

        # Penalize missing tool-required arguments (presence only; never check values).
        if tool in ALL_TOOLS:
            required = set(ALL_TOOLS[tool]["parameters"].get("required", []))
            if required:
                missing = sorted(k for k in required if k not in args)
                if missing:
                    score = max(0.0, score - 0.2 * len(missing) / len(required))
                    feedback.append(f"missing required arg(s) {', '.join(missing)}")

        if tool == "finish":
            want_status = constraints.get("status")
            if want_status and args.get("status") != want_status:
                score = max(0.0, score - 0.2)
                feedback.append(f"finish status expected {want_status!r}, got {args.get('status')!r}")
                if want_status == "impossible" and args.get("status") != "impossible":
                    unsafe = True
            if (gold_tool == "finish" and constraints.get("status") == "impossible"
                    and tool in PAID_TOOLS):
                unsafe = True

        if 0 < len(reasoning) <= 600:
            score += 0.1
        elif len(reasoning) > 600:
            feedback.append("reasoning is too long; keep it under 600 characters")

        return min(score, 1.0), ("; ".join(feedback) if feedback else "correct"), unsafe

    def gepa_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
        score, feedback, _ = score_case(gold, pred)
        return dspy.Prediction(score=score, feedback=feedback)

    float_metric = lambda gold, pred, trace=None: score_case(gold, pred)[0]

    print("RouteTool signature and metric ready.")
else:
    print("Skipped: DSPy is not available.")


## 5. Baselines

Three baselines on the same pilot cases:

- **B0 — production:** the real `SYSTEM_PROMPT` plus native OpenAI tool calling, exactly the
  code path in `agent.py`. This is the number that matters.
- **B1 — DSPy zero-shot:** `dspy.Predict(RouteTool)`.
- **B2 — DSPy CoT:** `dspy.ChainOfThought(RouteTool)`.

The pilot covers **2 cases per (suite, category) group** from the held-out test slices of the combined pool (routing, recovery, trajectory). Flip `RUN_BASELINES=True` to run it.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=config.llm_base_url or None,
    api_key=config.llm_api_key or "mock-key",
)


def production_route(case: dict):
    tools = [openai_tool_spec(t) for t in catalog_for_phase(case["phase"])]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"<task>\n{case['task']}\n</task>\n\n"
                f"<phase>\n{case['phase']}\n</phase>\n\n"
                f"<history>\n{case['history_summary'] or 'None'}\n</history>\n\n"
                "State 1 brief sentence of analytical reasoning in message content, then select exactly one next tool."
            ),
        },
    ]
    tool_choice = (
        {"type": "function", "function": {"name": "submit_plan"}}
        if case["phase"] == "plan"
        else "auto"
    )
    call_kwargs = dict(
        model=config.llm_model,
        messages=messages,
        tools=tools,
        tool_choice=tool_choice,
    )
    effort = getattr(config, "llm_reasoning_effort", "") or ("none" if "luna" in str(config.llm_model).lower() else "")
    if effort:
        call_kwargs["reasoning_effort"] = effort
    resp = client.chat.completions.create(**call_kwargs)
    msg = resp.choices[0].message
    if not msg.tool_calls:
        return type("Pred", (), {"tool_name": "", "arguments_json": "{}", "reasoning": msg.content or ""})()
    tc = msg.tool_calls[0]
    return type("Pred", (), {"tool_name": tc.function.name, "arguments_json": tc.function.arguments or "{}", "reasoning": msg.content or ""})()


def run_cases(program, cases: list[dict], max_cases: int | None = None) -> list[dict]:
    results = []
    for case in cases[: max_cases or len(cases)]:
        example = to_example(case)
        try:
            pred = program(
                task=example.task,
                phase=example.phase,
                history=example.history,
                tool_catalog=example.tool_catalog,
            )
        except Exception as exc:
            pred = type("Pred", (), {"tool_name": "", "arguments_json": "{}", "reasoning": f"ERROR: {exc}"})()
        score, feedback, unsafe = score_case(example, pred)
        results.append(
            {
                "id": case["id"],
                "phase": case["phase"],
                "gold": case["expected_next_tool"],
                "pred": str(getattr(pred, "tool_name", "") or ""),
                "args": parse_args(getattr(pred, "arguments_json", "{}")),
                "score": score,
                "feedback": feedback,
                "unsafe": unsafe,
                "reasoning": str(getattr(pred, "reasoning", "") or ""),
            }
        )
    return results


def summarize(results: list[dict], label: str) -> dict:
    if not results:
        return {"label": label, "n": 0}
    tool_match = statistics.mean(1.0 if r["pred"] in by_id[r["id"]]["acceptable_next_tools"] else 0.0 for r in results)
    mean_score = statistics.mean(r["score"] for r in results)
    unsafe_rate = statistics.mean(1.0 if r["unsafe"] else 0.0 for r in results)
    return {
        "label": label,
        "n": len(results),
        "tool_accuracy": round(tool_match, 4),
        "mean_score": round(mean_score, 4),
        "unsafe_rate": round(unsafe_rate, 4),
    }


def print_results(results: list[dict], label: str) -> None:
    print(f"\n=== {label} ===")
    for r in results:
        marker = "OK  " if r["pred"] in by_id[r["id"]]["acceptable_next_tools"] else "MISS"
        print(f"{marker} {r['id']:8s} gold={r['gold']:22s} pred={r['pred'] or '(none)':22s} score={r['score']:.2f}  {r['feedback'][:100]}")
    print(summarize(results, label))


BASELINE_RESULTS = {}
if RUN_BASELINES:
    # Suite-stratified pilot: 2 cases per (suite, category) group in sorted order,
    # covering routing, recovery, and trajectory suites from the held-out test slices.
    pilot = []
    per_group = collections.Counter()
    for case in sorted(TEST, key=lambda r: (r.get('suite', 'routing'), r['category'], r['id'])):
        group = (case.get('suite', 'routing'), case['category'])
        if per_group[group] < 2:
            pilot.append(case)
            per_group[group] += 1
    # production_route consumes the full case dict; run it directly.
    prod_results = []
    for case in pilot:
        pred = production_route(case)
        score, feedback, unsafe = score_case(case, pred)
        prod_results.append(
            {
                "id": case["id"], "phase": case["phase"], "gold": case["expected_next_tool"],
                "pred": pred.tool_name, "args": parse_args(pred.arguments_json),
                "score": score, "feedback": feedback, "unsafe": unsafe, "reasoning": pred.reasoning,
            }
        )
    BASELINE_RESULTS["production"] = prod_results
    print_results(prod_results, "B0 production SYSTEM_PROMPT + native tools")

    if DSPY_AVAILABLE:
        for label, use_cot in (("dspy_zero_shot", False), ("dspy_cot", True)):
            results = run_cases(Router(use_cot=use_cot), pilot)
            BASELINE_RESULTS[label] = results
            print_results(results, f"{label} (pilot={len(pilot)})")
    else:
        print("DSPy baselines skipped: dspy is not available.")

    table = [summarize(v, k) for k, v in BASELINE_RESULTS.items()]
    print("\nBaseline summary:")
    for row in table:
        print(row)
    print("\nPer-suite breakdown (production baseline):")
    for suite in ("routing", "recovery", "trajectory"):
        sub = [r for r in BASELINE_RESULTS.get("production", [])
               if by_id[r["id"]].get("suite", "routing") == suite]
        if sub:
            print(" ", summarize(sub, f"production/{suite}"))
    (OUT_DIR / "baseline_pilot.json").write_text(
        json.dumps({"summary": table, "results": BASELINE_RESULTS}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
else:
    print("Baselines disabled (RUN_BASELINES=False).")


## 6. GEPA compilation

`dspy.GEPA` uses a reflection LM to rewrite the router's instructions from metric feedback.
It returns a new program; the original is untouched. `auto="light"` keeps the search small.
`MAX_METRIC_CALLS` is a hard-ish budget; the exact accounting is DSPy's.

The compile output is a portable JSON artifact. It is **not** applied automatically.

In [ ]:
COMPILED = None
COMPILED_INSTRUCTIONS = ""
OPTIMIZATION_SUMMARY = {}

saved_compiled_path = OUT_DIR / "compiled_router.json"
if saved_compiled_path.is_file() and not RUN_OPTIMIZATION:
    print(f"Loading cached compiled router from {saved_compiled_path.name} (0 tokens spent)...")
    COMPILED = Router(use_cot=True)
    COMPILED.load_state(json.loads(saved_compiled_path.read_text(encoding="utf-8")))
    if (OUT_DIR / "compiled_router_instructions.md").is_file():
        COMPILED_INSTRUCTIONS = (OUT_DIR / "compiled_router_instructions.md").read_text(encoding="utf-8")
    if (OUT_DIR / "optimization_report.json").is_file():
        OPTIMIZATION_SUMMARY = json.loads((OUT_DIR / "optimization_report.json").read_text(encoding="utf-8"))
    print("Cached compiled router loaded successfully!")
elif RUN_OPTIMIZATION and DSPY_AVAILABLE:
    trainset = [to_example(r) for r in TRAIN]
    valset = [to_example(r) for r in DEV]
    student = Router(use_cot=True)

    before = run_cases(student, TEST)
    print("Before:", summarize(before, "student"))

    effort = getattr(config, "llm_reasoning_effort", "minimal") or "minimal"
    reflection_lm = dspy.LM(
        f"openai/{config.llm_model}",
        api_base=config.llm_base_url or None,
        api_key=config.llm_api_key or None,
        cache=True,
        reasoning_effort=effort,
    )

    optimizer_kwargs = dict(metric=gepa_metric, reflection_lm=reflection_lm, num_threads=2, max_metric_calls=MAX_METRIC_CALLS, seed=SEED)
    optimizer = dspy.GEPA(**optimizer_kwargs)
    COMPILED = optimizer.compile(student, trainset=trainset, valset=valset)

    after = run_cases(COMPILED, TEST)
    print("After :", summarize(after, "compiled"))

    try:
        for name, predictor in COMPILED.named_predictors():
            COMPILED_INSTRUCTIONS += f"## {name}\n\n{predictor.signature.instructions}\n\n"
    except Exception:
        for predictor in COMPILED.predictors():
            COMPILED_INSTRUCTIONS += f"{predictor.signature.instructions}\n\n"

    (OUT_DIR / "compiled_router.json").write_text(
        json.dumps(COMPILED.dump_state(), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (OUT_DIR / "compiled_router_instructions.md").write_text(COMPILED_INSTRUCTIONS, encoding="utf-8")
    OPTIMIZATION_SUMMARY = {
        "before": summarize(before, "student"),
        "after": summarize(after, "compiled"),
        "test_results": after,
    }
    (OUT_DIR / "optimization_report.json").write_text(
        json.dumps(OPTIMIZATION_SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8"
    )
elif RUN_OPTIMIZATION:
    print("Skipped: dspy is not available.")
else:
    print("Optimization disabled (RUN_OPTIMIZATION=False).")


## 7. Held-out evaluation and confusion matrix

This section evaluates whichever program is available, in order of preference: compiled
router → DSPy CoT → production baseline. It never trains on the test split.

The confusion matrix is the actionable output: it tells you which tool descriptions need
boundary work, not just an aggregate accuracy number.

In [ ]:
def confusion_table(results: list[dict]) -> None:
    if not results:
        print("No results.")
        return
    labels = sorted({r["gold"] for r in results} | {r["pred"] for r in results if r["pred"]})
    counts = collections.Counter((r["gold"], r["pred"]) for r in results)
    print("\nConfusion (gold -> predicted), non-zero off-diagonal:")
    for (gold, pred), n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
        if gold != pred:
            print(f"  {gold:24s} -> {pred or '(none)':24s} {n}")
    print("\nPer-tool precision / recall:")
    for tool in sorted({r["gold"] for r in results} | {r["pred"] for r in results if r["pred"]}):
        tp = sum(1 for r in results if r["gold"] == tool and r["pred"] == tool)
        fp = sum(1 for r in results if r["gold"] != tool and r["pred"] == tool)
        fn = sum(1 for r in results if r["gold"] == tool and r["pred"] != tool)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        print(f"  {tool:24s} P={precision:.2f} R={recall:.2f} (tp={tp} fp={fp} fn={fn})")


FINAL_RESULTS = []
if COMPILED is not None:
    FINAL_RESULTS = run_cases(COMPILED, TEST)
    print_results(FINAL_RESULTS, "compiled router on held-out test")
elif DSPY_AVAILABLE and RUN_BASELINES:
    FINAL_RESULTS = run_cases(Router(use_cot=True), TEST)
    print_results(FINAL_RESULTS, "DSPy CoT on held-out test")
else:
    print("No program evaluated. Enable RUN_OPTIMIZATION or RUN_BASELINES.")

confusion_table(FINAL_RESULTS)
if FINAL_RESULTS:
    (OUT_DIR / "test_results.json").write_text(
        json.dumps({"summary": summarize(FINAL_RESULTS, "final"), "results": FINAL_RESULTS}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

## 8. Tool-description tournament

DSPy does not rewrite tool docstrings. This section treats descriptions as a search space:

- **current** — the shipped description.
- **contract_first** — one-line purpose + `Use when` + `Do NOT use when` + cost.
- **contrastive** — names the nearest confusable tool and states the boundary.

The global comparison is cheap: three catalogs × the pilot set. The per-tool greedy pass is
more expensive and off by default. Keep the winner only if it improves the confusion pairs,
not just the aggregate.

In [ ]:
TOOL_NOTES = {
    "transcribe_audio": {
        "use_when": "speech in an audio or video file must become text or timestamped subtitles (.srt, .vtt). Handles files of any duration by auto-slicing <=30s chunks and writes subtitle deliverables directly to output_path.",
        "do_not_use_when": "the caller only needs duration/format metadata without speech-to-text, or the deliverable is spoken audio.",
        "neighbor": "inspect_media",
        "boundary": "inspect_media inspects audio duration/sample rate for free with zero ASR cost; transcribe_audio executes speech-to-text billing per minute.",
        "cost": "paid, about INR 0.10 per audio minute.",
    },
    "translate_text": {
        "use_when": "text already exists in context and must be translated into one of 28 supported Indian languages or English. Source language should be omitted for auto-detection.",
        "do_not_use_when": "the input is a file path or audio; or the deliverable is spoken audio (.wav); or the target language is unsupported non-Indic.",
        "neighbor": "synthesize_speech",
        "boundary": "translate_text produces translated text; synthesize_speech produces 24kHz WAV speech for text already in the target language.",
        "cost": "paid, about INR 0.20 per 10k tokens.",
    },
    "synthesize_speech": {
        "use_when": "the requested deliverable is spoken audio (.wav file) or reading text aloud. Text is already in the target Indian language or English.",
        "do_not_use_when": "the text requires translation first; or the caller only wants text/SRT saved to disk.",
        "neighbor": "write_text_file",
        "boundary": "synthesize_speech generates 24kHz WAV audio; write_text_file writes UTF-8 text and cannot create audio deliverables.",
        "cost": "paid, about INR 6.00 per 10k characters.",
    },
    "extract_document_text": {
        "use_when": "rendered documents (.pdf, .xps, .epub, .tiff) or raster images (.png, .jpg) must become text via Indic-OCR; or local whole .docx parsing.",
        "do_not_use_when": "the file is plain text (.txt, .md, .csv, .json, .srt), or bounded character pagination/slices are required (call read_text_file).",
        "neighbor": "read_text_file",
        "boundary": "extract_document_text returns full document/page text and has NO character pagination; read_text_file handles bounded slices and raw text.",
        "cost": "paid, about INR 0.20 per page image; DOCX fast path is free.",
    },
    "inspect_media": {
        "use_when": "checking audio duration, sample rate, image dimensions, format, or document page count before spending money on costly models.",
        "do_not_use_when": "the task needs actual text, transcripts, or translations extracted.",
        "neighbor": "extract_document_text",
        "boundary": "inspect_media checks metadata for free without OCR; extract_document_text runs OCR and bills per page image.",
        "cost": "free (zero API cost).",
    },
    "read_text_file": {
        "use_when": "reading plain text (.txt, .md, .csv, .json, .srt, .py) or bounded character pagination on ANY file including .docx using offset and limit.",
        "do_not_use_when": "the file is a PDF or raster image requiring OCR.",
        "neighbor": "extract_document_text",
        "boundary": "read_text_file slices characters using offset=start and limit=end-start+1; extract_document_text returns the full document and has no offset paging.",
        "cost": "free (zero API cost).",
    },
    "write_text_file": {
        "use_when": "saving translated markdown, summaries, reports, or preexisting text in context safely inside the authorized workspace root.",
        "do_not_use_when": "the deliverable is spoken audio (.wav) (use synthesize_speech), subtitles from an audio file (use transcribe_audio with output_format='srt'), or the destination path attempts directory traversal (..).",
        "neighbor": "synthesize_speech",
        "boundary": "write_text_file writes UTF-8 text strictly within workspace root; synthesize_speech generates WAV audio files.",
        "cost": "free (zero API cost).",
    },
    "detect_language": {
        "use_when": "text is already in context and you need to identify the script and plausible ISO codes using Unicode block analysis.",
        "do_not_use_when": "input is a file path or audio file (read or transcribe first); or language is already known from task context.",
        "neighbor": "read_text_file",
        "boundary": "detect_language analyzes characters already in memory for free; read_text_file is required first if content is in a file.",
        "cost": "free (zero API cost).",
    },
}


def _first_line(text: str) -> str:
    return text.splitlines()[0].strip() if text else ""


def contract_first(tool: dict) -> str:
    note = TOOL_NOTES.get(tool["name"], {})
    return (
        f"{_first_line(tool['description'])}\n\n"
        f"Use when: {note.get('use_when', 'the task matches this tool contract.')}\n"
        f"Do NOT use when: {note.get('do_not_use_when', 'another tool owns the deliverable.')}\n"
        f"Cost: {note.get('cost', 'unknown.')}"
    )


def contrastive(tool: dict) -> str:
    note = TOOL_NOTES.get(tool["name"], {})
    return (
        f"{contract_first(tool)}\n"
        f"Nearest confusable tool: {note.get('neighbor', 'none')}.\n"
        f"Boundary: {note.get('boundary', 'choose by deliverable type.')}"
    )


def build_overrides(variant: str) -> dict[str, str]:
    if variant == "current":
        return {}
    if variant == "contract_first":
        return {t["name"]: contract_first(t) for t in MCP_TOOLS}
    if variant == "contrastive":
        return {t["name"]: contrastive(t) for t in MCP_TOOLS}
    raise ValueError(variant)


def run_cases_with_overrides(program, cases: list[dict], variant: str) -> list[dict]:
    overrides = build_overrides(variant)
    results = []
    for case in cases:
        example = to_example(case, overrides)
        try:
            pred = program(
                task=example.task,
                phase=example.phase,
                history=example.history,
                tool_catalog=example.tool_catalog,
            )
        except Exception as exc:
            pred = type("Pred", (), {"tool_name": "", "arguments_json": "{}", "reasoning": f"ERROR: {exc}"})()
        score, feedback, unsafe = score_case(example, pred)
        results.append(
            {
                "id": case["id"], "phase": case["phase"], "gold": case["expected_next_tool"],
                "pred": str(getattr(pred, "tool_name", "") or ""), "score": score,
                "feedback": feedback, "unsafe": unsafe,
            }
        )
    return results


DESCRIPTION_TOURNAMENT = {}
if RUN_TOOL_DESC_TOURNAMENT and DSPY_AVAILABLE:
    base_program = COMPILED if COMPILED is not None else Router(use_cot=True)
    tournament_cases = DEV[:PILOT_N]
    for variant in ("current", "contract_first", "contrastive"):
        results = run_cases_with_overrides(base_program, tournament_cases, variant)
        DESCRIPTION_TOURNAMENT[variant] = {
            "summary": summarize(results, variant),
            "results": results,
            "overrides": build_overrides(variant),
        }
        print(variant, DESCRIPTION_TOURNAMENT[variant]["summary"])
    (OUT_DIR / "description_tournament.json").write_text(
        json.dumps(DESCRIPTION_TOURNAMENT, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    if RUN_FULL_TOOL_GREEDY:
        best_variant = max(DESCRIPTION_TOURNAMENT, key=lambda k: DESCRIPTION_TOURNAMENT[k]["summary"]["mean_score"])
        print("Global winner:", best_variant, "— starting greedy per-tool pass")
        current_overrides = build_overrides(best_variant)
        best_score = DESCRIPTION_TOURNAMENT[best_variant]["summary"]["mean_score"]
        for tool in ("transcribe_audio", "extract_document_text", "read_text_file",
                     "translate_text", "synthesize_speech", "write_text_file", "inspect_media",
                     "detect_language"):
            tool_obj = ALL_TOOLS[tool]
            candidate_overrides = dict(current_overrides)
            candidate_overrides[tool] = contrastive(tool_obj)
            # Reuse the variant runner by temporarily swapping the override map.
            results = []
            for case in tournament_cases:
                example = to_example(case, candidate_overrides)
                try:
                    pred = base_program(
                        task=example.task, phase=example.phase,
                        history=example.history, tool_catalog=example.tool_catalog,
                    )
                except Exception as exc:
                    pred = type("Pred", (), {"tool_name": "", "arguments_json": "{}", "reasoning": f"ERROR: {exc}"})()
                score, feedback, unsafe = score_case(example, pred)
                results.append({"id": case["id"], "phase": case["phase"], "gold": case["expected_next_tool"],
                                "pred": str(getattr(pred, "tool_name", "") or ""), "score": score,
                                "feedback": feedback, "unsafe": unsafe})
            score = statistics.mean(r["score"] for r in results)
            print(f"  {tool:24s} {best_score:.3f} -> {score:.3f}")
            if score > best_score:
                current_overrides = candidate_overrides
                best_score = score
        (OUT_DIR / "greedy_descriptions.json").write_text(
            json.dumps({"best_score": best_score, "overrides": current_overrides}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
elif RUN_TOOL_DESC_TOURNAMENT:
    print("Skipped: dspy is not available.")
else:
    print("Tool-description tournament disabled (RUN_TOOL_DESC_TOURNAMENT=False).")


## 9. Export candidates for review

Exports are deliberately **candidate artifacts**, not patches:

- `optimized_router_instructions.md` — the GEPA-compiled instruction text.
- `optimized_tool_descriptions.json` — the winning description variants.
- `optimization_report.md` — dataset hash, splits, baseline table, test results, confusions.
- `apply_diff.md` — a unified diff against the current `SYSTEM_PROMPT`.

Keep the safety block (English-only reasoning, untrusted-data quarantine, honest failure)
outside the optimized text. If you put it inside, GEPA may delete it when the metric does not
reward it.

In [ ]:
def _write(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    print("wrote", path.relative_to(recipe_root))


_write(
    EXPORT_DIR / "optimized_router_instructions.md",
    "# Candidate router instructions (review before applying)\n\n"
    + (COMPILED_INSTRUCTIONS or "(not compiled in this run)\n"),
)

winning_overrides = {}
desc_tournament = globals().get("DESCRIPTION_TOURNAMENT", {})
if desc_tournament:
    best_variant = max(desc_tournament, key=lambda k: desc_tournament[k]["summary"]["mean_score"])
    winning_overrides = desc_tournament[best_variant]["overrides"]
_write(
    EXPORT_DIR / "optimized_tool_descriptions.json",
    json.dumps(
        {
            "source": "description_tournament" if desc_tournament else "current",
            "winning_variant": max(desc_tournament, key=lambda k: desc_tournament[k]["summary"]["mean_score"]) if desc_tournament else "current",
            "overrides": winning_overrides,
        },
        ensure_ascii=False,
        indent=2,
    ),
)

report_lines = [
    "# DSPy Routing Lab Report",
    "",
    f"- Dataset: `{DATASET.relative_to(recipe_root)}`",
    f"- Cases: {len(rows)} (train {len(TRAIN)} / dev {len(DEV)} / test {len(TEST)})",
    f"- Dataset SHA-256: `{live_hash}`",
    f"- Model: `{config.llm_model}`",
    "",
    "## Baseline summary",
]
if BASELINE_RESULTS:
    for key, value in BASELINE_RESULTS.items():
        report_lines.append(f"- {summarize(value, key)}")
else:
    report_lines.append("- Not run in this session (RUN_BASELINES=False).")
report_lines += ["", "## Optimization"]
report_lines.append(f"- {OPTIMIZATION_SUMMARY.get('before') if OPTIMIZATION_SUMMARY else 'not compiled'}")
report_lines.append(f"- {OPTIMIZATION_SUMMARY.get('after') if OPTIMIZATION_SUMMARY else 'not compiled'}")
report_lines += ["", "## Held-out test"]
report_lines.append(f"- {summarize(FINAL_RESULTS, 'final') if FINAL_RESULTS else 'not evaluated'}")
report_lines += [
    "",
    "## Apply checklist",
    "1. Review the candidate instructions; keep the safety block outside them.",
    "2. Patch `SYSTEM_PROMPT` manually; never auto-apply.",
    "3. Re-run `pytest` and this notebook on the frozen test split.",
    "4. Re-run the lab whenever `LLM_MODEL` or the tool catalog changes.",
]
_write(EXPORT_DIR / "optimization_report.md", "\n".join(report_lines) + "\n")

if COMPILED_INSTRUCTIONS:
    diff = difflib.unified_diff(
        SYSTEM_PROMPT.splitlines(),
        COMPILED_INSTRUCTIONS.splitlines(),
        fromfile="current SYSTEM_PROMPT",
        tofile="candidate router instructions",
        lineterm="",
    )
    _write(EXPORT_DIR / "apply_diff.md", "```diff\n" + "\n".join(diff) + "\n```\n")
else:
    _write(EXPORT_DIR / "apply_diff.md", "No compiled instructions in this run.\n")


## 11. What to apply, and what not to trust

**Apply only after the held-out test improves.** The candidate router instructions are
model-specific: they were compiled against the configured LLM (`LLM_MODEL`) and this train/dev split. If
`LLM_MODEL` changes, rerun the lab.

**Do not move safety rules into the optimized block.** The English-only reasoning rule,
untrusted-data quarantine, honest-failure rule, and read sandbox must stay outside the
tunable instruction, or the metric must penalize violations explicitly.

**Do not auto-apply tool descriptions.** A description that wins on the benchmark can still
be wrong for a use case the benchmark does not cover. Read the diff, then patch `server.py`.

**Read the number correctly.** With 29 test cases one case is ~3.4 points, so a single
regression looks like a large accuracy drop and a single fix looks like a large gain. Prefer
the failure list over the percentage: which case ids flipped, and in which direction.

**What good looks like:**
- The four confusion pairs move in the right direction, not just the total.
- `unsafe_rate` stays at 0.0; if it rises, the metric is misaligned.
- The test set is untouched during iteration.

**Next experiments:**
1. Add cases for whichever confusion pair Section 7 shows is still failing.
2. If the router still misfiles arguments, extend the argument cases before touching GEPA.
3. Only after the offline router is stable, consider replacing the production loop with
   `dspy.ReAct`.
